# 9장 : 암과 싸워 이기기 위한 파이토치 활용

 + 커다란 문제를 잘게 쪼개서 해결하는 방법
 + 난해한 딥러닝 문제의 제약사항을 살펴보고 구조와 접근 방법 결정하기
 + 훈련 데이터 다운로드

9장에서는 우리가 사용하게 될 프로젝트와 환경, 데이터를 살펴보고, 앞으로 구현하게 될 프로젝트의 구조를 소개한다. 데이터 포맷, 데이터 소스, 그리고 우리 앞에 펼쳐진 문제 영역의 제약사항을 살펴보게 된다.

## 9.1 사용 사례 소개

현실에서는 예상한대로 딥러닝 모델 적용이 쉽지 않은 경우가 흔하다. 이러한 경우에 다룰 수 있는 도구와 방법들을, 악성 종양 자동 탐지 프로젝트를 통해 하나하나 알아나가보자. 우리는 환자의 흉부 CT 스캔을 입력으로 받아서 그것만으로 악성 종양을 자동으로 탐지하고자 한다.

구체적으로 말하자면 이번 프로젝트에서는 사람 몸통의 3차원 CT 스캔을 입력으로 받아서 악성 종양이 의심되는 경우 해당 위치를 출력한다.

왜 이러한 폐 종양 감지 문제를 설정했는가 하면, 이 문제가 아직 미해결 과제라는 점이다(이 책이 쓰인 시점 기준). 즉 파이토치를 사용해서 최첨단 프로젝트를 효과적으로 처리할 수 있다는 점을 밝히기 위함이다.

2부에서는 계속해서 폐 종양 탐지 문제에 집중하지만, 여기서 다루는 기술은 보편적이며 어떤 프로젝트를 수행하든 훈련을 위해 데이터를 조사하고 전처리하며 어떻게 표현할지를 학습하는 것은 중요하다. 우리가 폐 종양이라는 특정 상황에 대한 전처리를 다루더라도 기본 전제는 프로젝트의 성공적 수행을 위해 '우리는 무엇을 준비해야 하는가'일 것이다. 이와 마찬가지로, 훈련 루프를 설정하고, 올바른 성능 지표를 얻으며, 프로젝트 모델을 최종 애플리케이션으로 잘 연결하는 것은 모두 9장에서 14장까지 다룰 일반적인 기술이다.

## 9.2 대규모 프로젝트 준비

이번 프로젝트는 1부에서 배운 기술에 기반해 진행하게 되나, 입력으로는 3차원 데이터를 사용한다. 즉 파이토치 환경의 2차원에 대응하는 관련 도구는 더 이상 사용할 수 없다.

또한 8장과 달리, 바로 사용 가능한 데이터셋이 없다. 이를 해결하기 위해 11장까지는 우리가 다룰 두 가지 모델 아키텍처 설계를 시작조차 하지 않게 된다. 모델에 바로 꽂아 사용할 수 있는 훈련 샘플을 제공하는, 이미 구축된 라이브러리 형태의 표준 데이터가 없기 때문이다. 우리는 우리가 다룰 데이터에 대해 배우고 직접 구현도 해야 한다.

이외에도 GPU 리소스, 대용량의 저장 공간등 다양한 요구사항이 존재한다. 따라서 처음부터 종양이나 잠재적인 암의 징후를 찾기 위해 CT 스캔 전체를 들여다보지 말고, 대신 일련의 단순한 문제들을 풀고 이들을 조합하여 최종적으로 완성된 결과를 만들어보자. 문제를 덩어리로 나누고 분리하는 방법은 문제를 푸는 훌륭한 시작점이다. 막상 알고 보니 잘못된 접근이더라도, 우리는 문제를 성공적으로 풀어내기 위해 각 덩어리에서 얻은 내용들로 어떻게 재구성하면 될지에 대한 좋은 아이디어를 배우게 된다.

문제를 어떻게 잘게 나눌지에 대해 구체적으로 알아보기 전에 의학 분야에 대해 조금 알아둘 필요가 있다. 코드를 보면 무슨 일을 하는지 알 수 있긴 하지만 종양영상학에 대해 배워두면 그 일을 '왜' 하는지도 알 수 있다. 분야를 막론하고, 해결해야 할 문제 영역에 대해 공부하는 것은 매우 중요하다. 딥러닝은 강력하지만 마법이 아니므로, 의학과 같은 중차대한 문제에 대해 맹목적으로 적용하다가는 실패하고 말 것이다(이건 어떤 딥러닝이 아닌 어떤 방법론이든 마찬가지같다). 해당 분야에 대한 통찰력과 신경망의 동작에 대한 직관력을 잘 조합하여 제대로 훈련된 실험과 정제를 통해 정보를 얻음으로써, 우리는 쓸만한 해결책에 가까이 다가설 수 있을 것이다.

## 9.3 CT 스캔이란

CT 스캔은 단일 채널의 3차원 배열로 표현되는 3차원 엑스레이(X-ray)라 할 수 있다. 4장에서 살펴본 바대로, CT 스캔 데이터는 마치 흑백 PNG 이미지를 차곡차곡 쌓아놓은 것과 흡사하다.

$\textbf{복셀 (Voxel)}$

> 복셀(voxel)이란 우리에게 친숙한 2차원 픽셀의 3차원 버전으로 생각하면 된다. 면적이 아닌 공간 용적을 담고 있으며(용적 픽셀, VOlumetric piXEL의 준말이다) 데이터 필드를 표현하기 위해 3차원 격자로 배열된다. 각 차원 내에서는 거리를 측정할 수 있다. 통상적으로 복셀은 정육면체인 경우가 흔하지만 이 장에서 우리가 다룰 복셀은 직육면체의 모양을 띤다.

의학 데이터 외에도 $\text{유체}^{\text{fluid}}$ 시뮬레이션, 2차원 이미지로부터 재구성된 3차원 장면, 자율주행 자동차의 $\text{라이다}^{\text{LIDAR, light detection and ranging}}$ 데이터 등에서 복셀 데이터를 볼 수 있다. 우리가 사용할 API는 일반적인 여러 상황에도 적용가능하나, API를 좀 더 효과적으로 다루기 위해서는 데이터의 특성을 알아야 한다.

CT 스캔 내 각 복셀은 해당 위치에 있는 물체의 평균 질량 밀도를 나타내는 숫자 값을 가진다. 이 데이터들을 시각화하면 밀도가 높은 뼈나 금속 임플란트는 하얀색으로 보이고 밀도가 낮은 공기나 폐 조직은 검게 보인다. 지방이나 조직은 밝기가 다양한 회색으로 나타난다. 엑스레이와 유사하지만 본질적으로 다른 부분도 있다.

CT 스캔과 엑스레이의 근본적인 차이점은 엑스레이의 경우(조직이나 골밀도 같은) 3차원적인 강도를 2차원 평면에 $\text{투영}^{\text{projection}}$ 한 것이지만 CT 스캔은 데이터의 3차원 형태를 보존한다는 점이다. 따라서 그림 9.1처럼 회색 음영 형태의 다양한 방법으로 데이터를 가공할 수 있다(책 참조).

이와 같은 3차원 데이터 표현을 통해 우리는 관심 없는 조직을 숨겨서 대상의 '내부를 들여다 볼 수' 있다.

CT 스캔과 엑스레이의 또 다른 차이점은 데이터가 디지털 포맷이라는 점이다. CT라는 용어는 컴퓨터 $\text{단층촬영}^{\text{computed tomography}}$의 약어다. 즉 스캔과정의 원본 출력은 사람이 육안으로 구별할 만한 형태가 아니라서 컴퓨터를 사용해서 재해석해야 한다. 스캔 시 CT 스캐너 설정에 따라 출력 데이터는 크게 달라진다.

적절한 정보가 될지 모르겠지만 그림 9.3에서 살펴볼 수 있는 것으로 CT 스캐너가 머리부터 발에 이르는 축을 따라 거리를 측정하는 방법은 신체의 다른 두 가지 축과는 다르다는 사실을 알 수 있다. 환자는 실제로 축을 따라 이동한다! 이를 통해 복셀은 정육면체가 아님을 알 수 있고(또는 적어도 왜 아닌지 추측할 수 있으며), 이는 12장에서 데이터를 만져가며 다룰 때와 동일한 경우다. 이런 상황은 우리가 해당 분야를 이해하는 것이 왜 효과적인 해결안을 선택하는 데 도움이 되는지를 설명하는 적절한 예가 되기도 한다. 프로젝트를 시작하기 전에 데이터의 세부 사항을 구체적으로 조사할 필요가 있다.

## 9.4 프로젝트 : 엔드투엔드 폐암 진단기

$\textbf{엔드투엔드}^{\text{end-to-end}}$ 폐암 진단 솔루션
> 1. 데이터 로딩 (10장) : CT 스캔 데이터의 텐서화
> 2. 세그멘테이션 (13장) : 관심있는 복셀을 표시하기 위한 딥러닝 모델 학습 및 추론
> 3. 그룹화 (14장) : 추려진 복셀에 대해서 다시 작은 덩어리로 나누기
> 4. 분류 (11장, 12장) : 악성 결절인지, 양성 결절인지 분류하기 위한 딥러닝 모델 학습 및 추론
> 5. 결절 분석 및 진단 (14장)

$\text{결절}$

> 증식하는 세포로 이뤄진 조직 덩어리를 종양(tumor)이라고 한다. 종양은 양성(benign) 또는 악성(malignant) 으로 나뉘며, 악성일 경우 암(cancer)이라고도 한다. 폐에 있는 작은 종양(수 밀리미터에서 몇 센티미터)까지의 종양을 폐 결절(lung nodule)이라고 한다. 폐 결절의 약 40%는 악성인 것으로 알려져 있다. 따라서 최대한 초기에 잡아내는 것이 매우 중요하며, 이는 우리가 이 책에서 살펴보게 될 의학 영상 이미지에 달려 있다.


$\textbf{거인의 어깨 위에서}$

> 이와 같은 5가지 단계를 사용하는 것은 마치 거인의 어깨 위에 서 있는 것과 유사하다. 왜 그러한지는 14장에서 자세히 다룰 예정이다. 문제를 풀기 위해 이러한 프로젝트 구조가 왜 잘 동작하는지를 굳이 우리가 알아야 할 이유는 없지만, 비슷하게 만들어서 성공적인 결과를 보인 다른 사례들에 우리는 의탁하게 된다. 다른 도메인으로 전환할 때 실행 가능한 접근법을 찾으려는 실험을 해보자. 

> 하지만 동시에, 해당 분야에서의 과거 노력으로부터, 그리고 유사한 영역에서 효과가 있었으며 잘 전환될 만한 것들을 발견한 이들로부터 항상 배우려고 노력해보자. 외부로 눈을 돌려 다른 사람들은 어떤 연구를 했는지 찾아보고 그 결과를 벤치마크 자료로 사용하자. 그리고 아무것도 모른 채로 다른 이의 코드를 가져와 돌려보는 일은 없어야 한다(매우 중요). 스스로 진도를 내고자 결과를 사용한다면 돌리고 있는 코드를 완전히 이해해야 하기 때문이다.

다른 프로젝트를 하게 되더라도 여전히 데이터나 문제 도메인에 대해 조사해야 한다. 만약 위성지도 제작에 관심이 있다면 다음 프로젝트는 궤도에서 찍은 지구 이미지를 사용해야 할 것이다. 수집한 파장에 대해 물어봐야 할 수도 있다. 일반 RGB 이미지만 있으면 될까? 아니면 특이한 정보가 필요할까? 적외선이나 자외선이 필요할지도 모른다. 여기에 하루 중 몇 시에 촬영했는지에 따라 정보에 영향을 줬을 수도 있고, 이미지에서 나타나는 위치가 위성 바로 아래쪽이 아니라서 그림이 왜곡됐을 수도 있으며, 이미지 보정이 필요할 수도 있다.

잠재적인 최적화나 개선을 꾀하려면 데이터에 대한 직관력을 활용할 수 있어야 한다. 이런 점은 딥러닝 프로젝트에서 일반적으로 요구되는 것으로, 2부에서도 직관력을 활용하는 연습을 할 것이다. 지금 우리가 접근하려는 방식이 직관적으로 괜찮은가? 너무 복잡해보이지는 않는가?

### 9.4.1 신경망이 동작할 때까지 데이터를 던져넣을 수 없는 이유

이 문제가 어려운 이유는, CT 스캔은 일단 대부분의 경우 악성 종양이 있더라도 99.9999%의 복셀은 암 세포가 아니다. 비율로 따지면 고해상도TV(HD) 어딘가에 색이 상한 2개 픽셀 정도에 해당한다(반도체 다이의 불량 부분과 비슷한듯하다. 나중에 이걸 프로젝트화해봐도 될 것 같다).

문제를 풀기 위한 우리의 접근 방식은 최종 목표에 맞는 최적화로 엔드투엔드 기울기 역전파를 사용하진 않을 것이다. 대신 문제의 부분 부분을 최적화하는 방식으로 동작한다. 우리의 세그멘테이션 모델과 분류 모델이 서로 협력하게끔 훈련된 것이 아니기 때문이다. 따라서 솔루션의 효율성이 최대가 되지는 않지만, 배우는 입장에서는 더 낫다고 생각한다(나중에 설계 다시 해보기).

또 한 번에 한 단계씩 초점을 맞추면, 이 안에서 요구하는 기술에만 집중할 수 있는 효과도 있다. 두 개의 모델 각각은 정확하게 하나의 작업을 수행하는 데 집중한다. 방사선 전문의가 CT 단면을 하나하나 검토하는 것처럼, 범위가 정리되어 있으면 작업을 훈련시키기도 쉬워진다. 그리고 우리는 데이터를 자유롭게 조작할 수 있는 도구도 제공하고자 한다. 특정 위치에 대해 상세하게 볼 수 있는 확대 기능이 있다면, 전체 이미지를 한 번에 봐야 할 때보다 모델을 훈련시킬 때 전체적인 생산성이 크게 좋아진다. 세그멘테이션 모델은 전체 이미지를 사용하게 되어 있지만 이를 잘 구성해서 분류 모델은 관심 영역을 확대한 데이터를 사용하게 할 예정이다.

3단계의 그룹화는 데이터를 만들고, 4단계의 분류는 이 데이터를 사용하는데, 여기서 데이터란 종양에 대한 연속적인 CT 단면을 말한다. 우리는 4단계의 모델이 이러한 이미지를 식별할 수 있도록 훈련시킬 것이다. 그리고 5단계의 모델은 이 데이터가 양성인지 악성인지를 분류한다. 전체 CT를 살펴보기 보다, 종양에 대한 CT 단면만을 살펴보는 문제가 훨씬 더 쉬운 문제임에 틀림없다.

우리는 10장에서 데이터를 읽는 1단계를 수행하고, 11장과 12장에서 결절을 분류하는 문제에 집중할 것이다. 그리고 나서 13장에서는 다시 2단계(결절 후보들을 찾아 세그멘테이션하기)로 돌아오고 14장에서는 3단계(그룹화)와 5단계(결절 분석 및 진단)를 모두 합쳐 엔드투엔드 프로젝트로 완성하여 2부를 마무리할 것이다.

### 9.4.2 결절이란 무엇인가

결절이란, 글자 그대로 누군가의 폐 내부에서 나타날 수 있는 무수한 돌기나 덩어리를 말한다. 결절에 따라 환자의 건강 면에서 문제가 되는 것도 있지만 그렇지 않은 경우도 있다.

정확한 정의에 따르면, 결절의 크기는 3센티미터 이하여야 하며, 이보다 클 경우에는 $\text{폐 종괴}^{\text{lung mass}}$ 라 부른다. 다만 이 기준은 임의적이고 3센티미터보다 크든 작든 동일한 코드 경로를 사용하게 되므로, 편의상 우리 책에서는 동일한 해부학적 구조라면 모두 결절이라고 부르고자 한다. 결절은 양성 종양 또는 악성 종양으로 밝혀질 수 있다. 방사선학 관점에서 결절은 감염이나 염증, 혈액 공급 이상, 기형적 혈관, 종양 이외의 질환 등 다양한 원인으로 유발된 덩어리라 할 수 있다.

여기서 중요한 핵심은, 우리가 찾아내려는 암은 항상 결절의 형태를 띨 것이며, 이 결절은 밀도가 매우 낮은 폐 조직에 매달려 있거나 폐 벽에 붙어 있을 것이라는 점이다. 따라서 우리의 분류기는 모든 조직을 검사할 필요 없이 결절에만 집중하면 된다. 이런 식으로 예상 입력 범위를 좁히면 분류기가 작업을 학습하는 데 도움이 된다.

이 부분은 딥러닝에서 보편적으로 사용하는 기술이기도 하다. 다만 잘 알고 사용해야 한다. 이런 방법으로 도움을 얻기 위해서는 자신이 적용하려는 분야에 대한 이해도가 높아야 한다.

### 9.4.3 데이터 소스 : LUNA 그랜드 챌린지

우리가 보는 CT 스캔은 $\text{LUng Nodule Analysis}$ (폐 결절 분석) 그랜드 챌린지에서 가져온 것이다. LUNA 그랜드 챌린지는 (다량의 폐 결절이 있는) 환자의 CT 스캔 자료를 고품질로 레이블 작업한 공개 데이터셋으로서 데이터 분류기 성능에 대한 공개 랭킹을 제공한다.

이 책에서 우리는 LUNA 2016 데이터셋을 사용한다. LUNA 사이트(https://luna16.grand-challenge.org/Description/)에는 두 종류의 도전 기록을 볼 수 있다. 하나는 '결절 탐지'로 대략 우리의 첫 번째 세그멘테이션 단계에 해당하며, 다른 기록은 '거짓 양성 감소'로 우리의 3번째 분류 단계와 유사하다. 사이트를 보면 '잠재적 결절의 위치'에 대한 내용이 있는데, 이는 우리가 13장에서 다룰 내용과 유사하다.

### 9.4.4 LUNA 데이터 다운로드

LUNA 데이터는 대략 60GB 정도의 압축된  데이터로, 압축을 풀면 120GB의 공간을 차지한다(지금은 업데이트 되서 그런지 대략 130GB정도). 이외에 추가로, CT 전체 데이터를 읽는 대신 필요한 부분만 읽도록 데이터를 작은 덩어리로 나눠둘 100GB 정도의 캐시 공간이 필요하다.

## 9.5 결론

지금까지 두 가지를 살펴봤다.

  1. 폐암 진단 프로젝트를 큰 맥락에서 이해해봄.
  2. 2부 학습을 위한 프로젝트의 방향과 구조에 대한 개요 파악

프로젝트가 동작하는 전문 영역에 대한 이해를 높이는 것은 매우 중요하며, 이와 같이 마쳐둔 설계는 앞으로 프로젝트를 진행하며 훌륭하게 제값을 할 것이다. 다음 10장에서 데이터 로딩을 구현하기 시작하면서 바로 여기서 공부한 덕을 볼 것이다.

## 9.6 핵심 요약

  - 우리는 암으로 추정되는 결절을 발견하기 위해 데이터 로딩, 세그멘테이션, 그룹화, 분류, 결절 분석 및 진단으로 이어지는 다섯 단계의 접근 방식을 사용한다.
  - 프로젝트를 작고 독립적인 하위 프로젝트로 나누면 각각의 하위 프로젝트를 좀 더 쉽게 설명할 수 있다. 이 책의 프로젝트와 목표가 다른 향후 프로젝트를 진행할 때는 지금과는 다른 방식의 접근법들이 적절할 수도 있다.
  - CT 스캔은 대략적으로 3,200만 개의 복셀을 가진 3차원의 밀도 데이터 배열로서, 우리가 탐지하려는 결절보다 약 100만 배 정도 크다. 수행하려는 작업에 맞게 CT 스캔을 잘라내는 모델에 집중하면 훈련을 통해 꽤 괜찮은 결과를 더 쉽게 얻을 수 있다.
  - 데이터 처리 루틴을 작성하기 전에 데이터 자체에 대해 잘 이해하면 데이터의 중요한 특징을 왜곡하거나 누락할 가능성을 줄일 수 있다. CT 스캔 데이터는 일반적으로 정방형의 복셀 데이터가 아니다. 현실 세계에서의 위치 정보를 배열의 인덱스로 변경하려면 별도의 변환 작업이 필요하다. CT 스캔의 밀도는 대략적으로 대상의 밀도와 일치하지만 특수한 단위를 사용한다.
  - 프로젝트에서 사용하는 주요 개념을 이해하고 설계에서 이 부분이 잘 반영되게 하는 것은 매우 중요하다. 우리 프로젝트의 대부분은 폐 결절을 중심으로 문제를 풀어가게 된다. 폐 결절은 폐에 있는 작은 덩어리로서, 유사한 모양을 띠는 다른 많은 구조들과 함께 CT상에서 발견될 수 있다.
  - 우리는 모델 훈련을 위해 LUNA 그랜드 챌린지 데이터를 사용한다. LUNA 데이터에는 CT 스캔데이터뿐만 아니라, 분류와 그룹화를 위해 사람이 직접 레이블 작업을 한 출력값이 포함된다. 고품질의 데이터를 확보하는 것은 프로젝트를 성공시키는 데 큰 영향을 미친다.